In [ ]:
#housekeeping

from google.colab import drive
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from PIL import Image

from tensorflow.keras.models import load_model
from IPython.display import clear_output

In [ ]:
#mount drive and save dataset directory
drive.mount('/content/drive')
dataset = '/content/drive/My Drive/PVdatasetUpdated'

Mounted at /content/drive


In [ ]:
#convert file of files of images into a single, usable dataframe of labeled data

#gather all images and their labels into one dataframe
image_paths = []
labels = []
unique_labels = []
for plant_type in os.listdir(dataset):
  plant_path = os.path.join(dataset, plant_type)
  if not os.path.isdir(plant_path):
    continue

  for partition in os.listdir(plant_path):
    partition_path = os.path.join(plant_path, partition)
    if not os.path.isdir(partition_path):
      continue

    for label in os.listdir(partition_path):
      label_path = os.path.join(partition_path, label)
        #get the names of folders since they are labels: Healthy, Rust, etc.
      if not os.path.isdir(label_path):
        continue

      for img_file in os.listdir(label_path):
        img_file_path = os.path.join(label_path, img_file)
        if os.path.isfile(img_file_path) and img_file.lower().endswith(('jpg', 'jpeg', '.png')):
          #add full path to img_file to image_paths list
            #and label of the image (taken from folder name) to labels list
          image_paths.append(img_file_path)
          if label not in labels:
            unique_labels.append(label)
          labels.append(label)

labels_as_indexes = []
for label in labels:
  label_index = unique_labels.index(label)
  labels_as_indexes.append(label_index)

#make new labels based on
df = pd.DataFrame({
    "image": image_paths,
    "label": labels,
    "label_index": labels_as_indexes
})

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

#build new dataframe with images labelled with lists of symptoms
symptoms_json_path = '/content/drive/My Drive/plantVis3/disease_symptoms.json'
disease_to_symptoms_df = pd.read_json(symptoms_json_path)

def get_symptoms(df, disease):
    row = df.loc[df["disease"] == disease, "symptoms"]
    return row.iloc[0] if not row.empty else None

symptoms_labels = []
unique_symptoms = []

# Build list of symptom lists for each disease label
for label in df.label:      # <-- assumes df.label contains disease names
    symptoms = get_symptoms(disease_to_symptoms_df, label)
    symptoms_labels.append(symptoms)
    if symptoms is None:
        continue
    for symp in symptoms:
        if symp not in unique_symptoms:
            unique_symptoms.append(symp)

print(unique_symptoms)
print(len(unique_symptoms))

symptoms_labels_encoded = []
for symps in symptoms_labels:
    # create vector of zeros initially
    row_vec = [0] * len(unique_symptoms)
    if symps is not None:
        for s in symps:
            idx = unique_symptoms.index(s)
            row_vec[idx] = 1
    symptoms_labels_encoded.append(row_vec)

image_to_symptoms_df = pd.DataFrame({
    "image": image_paths,
    "symptoms": symptoms_labels,
    "symptoms_encoded": symptoms_labels_encoded
})

#35 unique symptoms
#Cedar Apple Rust,Black Rot,Healthy,Apple Scab,Bacterial Spot,Powdery Mildew,Cercospora Leaf Spot,Common Rust,Northern Leaf Blight,Leaf Blight,Esca (Black Measles),Leaf Scorch,Early Blight,Late Blight,Septoria Leaf Spot,Yellow Leaf Curl Virus,


['White-Mold', 'Circular', 'Black', 'Rings', 'Brown', 'Oily', 'Yellow', 'Greasy', 'Scorched', 'Purple', 'Brown-Lesions', 'Watery', 'Halo', 'Spots', 'Tan', 'Sunken', 'Black-Lesions', 'Black-Veins', 'Rotting', 'Wilt', 'Blotchy', 'Discolored-Streaks', 'Collapsed', 'Fuzzy', 'Shriveled', 'Distortion', 'Sunscald', 'Powdery', 'Elliptical-Lesions', 'Parallel-Lesions', 'Green-Lesions', 'Spores', 'Yellow-Lesions', 'Black-Specks', 'Discolored']
35


In [ ]:
#housekeeping value assignments and data shuffling

#define class weights
encoded_array = np.stack(image_to_symptoms_df.symptoms_encoded.values)  # shape (num_samples, 35)
class_counts = encoded_array.sum(axis=0)  # total positive instances per class
total_instances = encoded_array.shape[0]
class_weights = total_instances / class_counts  # fraction of positives
class_weights = class_weights / np.max(class_weights) #normalize

class_weights = np.array(class_weights, dtype=np.float32)  # ensure float32
class_weights = tf.constant(class_weights)  # convert to tensor

#split new symptoms dataframe into training, validation, and testing datasets
train_df, test_df = train_test_split(image_to_symptoms_df, test_size=0.2, stratify=image_to_symptoms_df["symptoms_encoded"], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df["symptoms_encoded"], random_state=42)

#turn the new dataframes into seperate lists of X input images and Y labels
X_train, y_train = train_df["image"].tolist(), train_df["symptoms_encoded"].tolist()
X_val, y_val = val_df["image"].tolist(), val_df["symptoms_encoded"].tolist()
X_test, y_test = test_df["image"].tolist(), test_df["symptoms_encoded"].tolist()

In [ ]:
#visualize data
rand = np.random.randint(32)

print("X_train\nLength:\t", len(X_train), "\nExample:\t", X_train[rand], "\nType:\t", type(X_train), "\n")
print("X_val\nLength:\t", len(X_val), "\nExample:\t", X_val[rand], "\nType:\t", type(X_val), "\n")
print("X_test\nLength:\t", len(X_test), "\nExample:\t", X_test[rand], "\nType:\t", type(X_test), "\n")

print("y_train\nLength:\t", len(y_train), "\nExample:\t", y_train[rand], "\nType:\t", type(y_train), "\n")
print("y_val\nLength:\t", len(y_val), "\nExample:\t", y_val[rand], "\nType:\t", type(y_val), "\n")
print("y_test\nLength:\t", len(y_test), "\nExample:\t", y_test[rand], "\nType:\t", type(y_test), "\n")


X_train
Length:	 42950 
Example:	 /content/drive/My Drive/PVdatasetUpdated/Cherry/Train/Healthy/e488b116-6b54-43e7-8b4e-232eb434941b___JR_HL 9425_180deg.JPG 
Type:	 <class 'list'> 

X_val
Length:	 10738 
Example:	 /content/drive/My Drive/PVdatasetUpdated/Strawberry/Train/Leaf Scorch/e6a58ef4-f85e-4a73-a5f9-b1ba93d98fd1___RS_L.Scorch 1035.JPG 
Type:	 <class 'list'> 

X_test
Length:	 13423 
Example:	 /content/drive/My Drive/PVdatasetUpdated/Tomato/Train/Early Blight/e6645f63-afc8-416a-a4cb-0f5c719476ae___RS_Erly.B 6419_180deg.JPG 
Type:	 <class 'list'> 

y_train
Length:	 42950 
Example:	 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] 
Type:	 <class 'list'> 

y_val
Length:	 10738 
Example:	 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] 
Type:	 <class 'list'> 

y_test
Length:	 13423 
Example:	 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
 #build the model from model2

'''
#import old model, select layers to reuse
old_model_path="/content/drive/My Drive/plantVis2/plantVis2.keras"
old_model=keras.models.load_model(old_model_path)

old_model.summary()

#from keras.utils import plot_model
#plot_model(old_model)#, show_shapes=True, show_layer_names=True)


layer = old_model.layers[-2]
print("Name:\t", layer.name)
print("Input:\t", layer.input)   # tensor
print("Output:\t", layer.output)  # tensor
print("Input Shape:\t", layer.input.shape)
print("Output Shape:\t", layer.output.shape)


#get second to last layer
old_base_output = old_model.layers[-2].output #-2
#add dropout
dropout_layer = keras.layers.Dropout(0.3)(old_base_output)

#make new output layer for 35 possible classes
#output_layer=keras.layers.Dense(35, activation="sigmoid")(dropout_layer)

#new output layer for 35 possible classes using logits
logits_layer = keras.layers.Dense(35, name="logits_layer")(dropout_layer)

#build new model
cnn = keras.Model(inputs=old_model.inputs, outputs=logits_layer)

cnn.summary()
#'''

'\n#import old model, select layers to reuse\nold_model_path="/content/drive/My Drive/plantVis2/plantVis2.keras"\nold_model=keras.models.load_model(old_model_path)\n\nold_model.summary()\n\n#from keras.utils import plot_model\n#plot_model(old_model)#, show_shapes=True, show_layer_names=True)\n\n\nlayer = old_model.layers[-2]\nprint("Name:\t", layer.name)\nprint("Input:\t", layer.input)   # tensor\nprint("Output:\t", layer.output)  # tensor\nprint("Input Shape:\t", layer.input.shape)\nprint("Output Shape:\t", layer.output.shape)\n\n\n#get second to last layer\nold_base_output = old_model.layers[-2].output #-2\n#add dropout\ndropout_layer = keras.layers.Dropout(0.3)(old_base_output)\n\n#make new output layer for 35 possible classes\n#output_layer=keras.layers.Dense(35, activation="sigmoid")(dropout_layer)\n\n#new output layer for 35 possible classes using logits\nlogits_layer = keras.layers.Dense(35, name="logits_layer")(dropout_layer)\n\n#build new model\ncnn = keras.Model(inputs=old_mo

In [ ]:
#define some functions and a class
  #load_images_from_paths()
  #close_images_from_paths()
  #slice_wrap()
  #train_one_batch()
  #train_in_batches()
  #test_in_batches()
  #weighted_bce_loss()
  #class WeightedBCELoss
  #f1_metric()

def load_images_from_paths(path_list):
    images = []
    target_size = (256, 256)
    for path in path_list:
        img = Image.open(path).convert("RGB")
        img = img.resize(target_size)
        img_array = np.array(img) / 255.0
        images.append(img_array)
    return np.array(images)

def close_images_from_paths(path_list):
  for path in path_list:
    Image.close(path)

def slice_wrap(x, start, end):
    n = len(x)
    return [x[(start + i) % n] for i in range(end-start)]

def train_one_batch(cnn, batch_size, number_of_batches, batch_num, x_train, x_val, y_train, y_val):
  print("batch", batch_num, " of ", number_of_batches)

  start = batch_size*batch_num
  end = start+batch_size

  batch_xt = x_train[start:end]
  batch_yt = y_train[start:end]
  batch_xv = slice_wrap(x_val, start, end)
  batch_yv = slice_wrap(y_val, start, end)


  batch_train_img = load_images_from_paths(batch_xt)
  batch_val_img = load_images_from_paths(batch_xv)

  '''
  print("batch_yt len:", len(np.array(batch_yt))) #64
  print("batdh_yt element len:", len(np.array(batch_yt[0])))  #35
  print("batch_yv len:", len(np.array(batch_yv))) #64
  print("batdh_yv element len:", len(np.array(batch_yv[0])))  #35
  '''

  history = cnn.fit(
      batch_train_img,
      np.array(batch_yt),
      validation_data=(batch_val_img, np.array(batch_yv))
  )
  return history

def train_in_batches(cnn, batch_size, number_of_batches, x_train, x_val, y_train, y_val):
  for batch_num in range(number_of_batches):
    train_one_batch(cnn, batch_size, batch_num, x_train, x_val, y_train, y_val)

def test_in_batches(cnn, batch_size, x_test, y_test):
  number_of_batches = int(np.ceil( len(x_test)/batch_size ))

  for batch_num in range(number_of_batches):
    start = batch_size*batch_num
    end = start + batch_size

    batch_x = x_test[start:end]
    batch_y = y_test[start:end]

    batch_img = load_images_from_paths(batch_x)

    evaluation = cnn.evaluate(batch_img, np.array(batch_y))

    loss, accuracy = evaluation[0], evaluation[1]
    metrics_frame = pd.DataFrame({
      "Testing Accuracy": [accuracy],
      "Testing Loss": [loss]
    })
    metrics_frame.to_csv(metrics_history_path, mode="a", header=False, index=False)

#def bce_claculate(y_t, y_p):


@tf.keras.utils.register_keras_serializable()
def weighted_bce_loss(y_true, y_pred):
    #print("y_true.shape:", y_true.shape)  #32, 35
    #print("y_pred.shape:", y_pred.shape)  #32, 35

    #bce = tf.keras.losses.binary_crossentropy(y_true, y_pred, from_logits=True)
    #bce = bce_fn(y_true, y_pred)
    #bce = bce_calculate(y_true, y_pred)
    bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true, logits=y_pred)
    #print("bce.shape:", bce.shape)

    # Safety check
    if bce.shape[-1] != class_weights.shape[0]:
        raise ValueError(
            f"Shape mismatch: BCE has shape of {bce.shape}, but "
            f"class_weights has shape of {class_weights.shape}."
        )

    # Apply per-class weights
    weighted_bce = bce * class_weights  # broadcast across batch

    # Reduce to single scalar loss
    return tf.reduce_mean(weighted_bce)

@tf.keras.utils.register_keras_serializable()
class WeightedBCELoss(keras.losses.Loss):
    def __init__(self, class_weights):
        super().__init__()
        self.class_weights = tf.constant(class_weights, dtype=tf.float32)

    def call(self, y_true, y_pred):
        bce = keras.losses.binary_crossentropy(y_true, y_pred, from_logits=True)
        weighted_bce = bce * self.class_weights
        return tf.reduce_mean(weighted_bce)

@tf.keras.utils.register_keras_serializable()
def f1_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)


    # If using logits, convert to probabilities
    y_pred = tf.nn.sigmoid(y_pred)

    # Convert probabilities to 0/1 predictions
    y_pred_bin = tf.cast(y_pred > 0.5, tf.float32)

    tp = tf.reduce_sum(y_true * y_pred_bin, axis=0)
    fp = tf.reduce_sum((1 - y_true) * y_pred_bin, axis=0)
    fn = tf.reduce_sum(y_true * (1 - y_pred_bin), axis=0)

    tp = tf.cast(tp, tf.float32)
    fp = tf.cast(fp, tf.float32)
    fn = tf.cast(fn, tf.float32)

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)

    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return tf.reduce_mean(f1)

In [ ]:
#compile and save the model

'''
#freeze whole model, with excetpions
for layer in cnn.layers:
  layer.trainable = False
cnn.layers[-2].trainable = True
cnn.layers[-1].trainable = True

cnn.compile(optimizer='adam',
            loss=weighted_bce_loss,
            metrics=['accuracy',
                     tf.keras.metrics.Precision(name='precision'),
                     tf.keras.metrics.Recall(name='recall'),
                     f1_metric]
)


cnn.save("/content/drive/My Drive/plantVis3/model3.keras")
#'''

'\n#freeze whole model, with excetpions\nfor layer in cnn.layers:\n  layer.trainable = False\ncnn.layers[-2].trainable = True\ncnn.layers[-1].trainable = True\n\ncnn.compile(optimizer=\'adam\',\n            loss=weighted_bce_loss,\n            metrics=[\'accuracy\',\n                     tf.keras.metrics.Precision(name=\'precision\'),\n                     tf.keras.metrics.Recall(name=\'recall\'),\n                     f1_metric]\n)\n\n\ncnn.save("/content/drive/My Drive/plantVis3/model3.keras")\n#'

In [ ]:
model_path = "/content/drive/My Drive/plantVis3/model3.keras"
cnn = load_model(model_path,
                          custom_objects={
                          'weighted_bce_loss': weighted_bce_loss,
                          'f1_metric': f1_metric
                          })
#freeze whole model, with excetpions
for layer in cnn.layers:
  layer.trainable = True#False
#cnn.layers[-2].trainable = True
#cnn.layers[-1].trainable = True

cnn.compile(optimizer='adam',
            loss=weighted_bce_loss,
            metrics=['accuracy',
                     tf.keras.metrics.Precision(name='precision'),
                     tf.keras.metrics.Recall(name='recall'),
                     f1_metric]
)


cnn.save("/content/drive/My Drive/plantVis3/model3.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:


print("cnn summary:")
cnn.summary()

print("\nfirst five labels:\n", y_train[:5])

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
for img, label in train_ds.take(1):
    print(img[0].numpy().shape, label[0].numpy())

cnn summary:


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 246016)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    15,745,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits_layer (Dense)            │ (None, 35)             │         2,275 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,300,267 (180.44 MB)

 Trainable params: 15,766,755 (60.15 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 31,533,512 (120.29 MB)


first five labels:
 [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1], [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


InvalidArgumentError: {{function_node __wrapped__StridedSlice_device_/job:localhost/replica:0/task:0/device:CPU:0}} Attempting to slice scalar input. [Op:StridedSlice] name: strided_slice/

In [ ]:
def parse_image_path(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)   # or decode_png
    img = tf.image.resize(img, [256, 256])
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.cast(label, tf.float32) # Explicitly cast label to float32
    return img, label

class SavePeriodic(tf.keras.callbacks.Callback):
    def __init__(self, steps_interval=200):
        super().__init__()
        self.steps_interval = steps_interval
        self.batch_losses = []
        self.batch_accs = []

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}

        # Store per-batch metrics
        if "loss" in logs:
            self.batch_losses.append(logs["loss"])
        if "accuracy" in logs:
            self.batch_accs.append(logs["accuracy"])

        # Autosave + plot periodically
        if batch % self.steps_interval == 0 and batch > 0:
            self.model.save("autosave.keras")
            print(f"Autosave at batch {batch}")

            # Plot updated graphs
            self.plot_metrics()

    def plot_metrics(self):
        plt.figure(figsize=(10, 4))

        # Loss curve
        plt.subplot(1, 2, 1)
        plt.plot(self.batch_losses)
        plt.title("Batch Loss History")
        plt.xlabel("Batch")
        plt.ylabel("Loss")

        # Accuracy curve (only if available)
        plt.subplot(1, 2, 2)
        if self.batch_accs:
            plt.plot(self.batch_accs)
            plt.title("Batch Accuracy History")
            plt.xlabel("Batch")
            plt.ylabel("Accuracy")

        plt.tight_layout()
        plt.show()

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (train_ds.shuffle(4096)
            .map(parse_image_path, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(64)
            .prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(parse_image_path).batch(64)

print("cnn summary:")
cnn.summary()

print("\nfirst five labels:\n", y_train[:5])

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
for path, label in train_ds.take(1):
    print("PATH:", path.numpy())
    print("LABEL:", label.numpy())

#cnn.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[SavePeriodic(200)])


cnn summary:


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 246016)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    15,745,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits_layer (Dense)            │ (None, 35)             │         2,275 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,300,267 (180.44 MB)

 Trainable params: 15,766,755 (60.15 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 31,533,512 (120.29 MB)


first five labels:
 [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1], [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
PATH: b'/content/drive/My Drive/PVdatasetUpdated/Cherry/Train/Healthy/e488b116-6b54-43e7-8b4e-232eb434941b___JR_HL 9425_180deg.JPG'
LABEL: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
'''
The Plan
freeze everything but the last layer and train, see where it's at after an epoch, might need 3 to 5 epochs
  look for a loss platue as a sign to unfreeze more
unfreeze dense layer, train at least one more epoch, likely 5 to 10 more
after loss stops improving, unfreeze all and continue training
  change to smaller learning rate


model_path = "/content/drive/My Drive/plantVis3/model3.keras"
train_index_path = "/content/drive/My Drive/plantVis3/model3trainingIndex.txt"
train_metrics_history_path = "/content/drive/My Drive/plantVis3/train_metrics_history2.csv"

batch_size = 32
number_of_batches = int(np.ceil( len(X_train)/batch_size ))
batches_at_once = 15
batch_num = 0

train_acc = []
train_recall=[]
train_precision=[]
train_F1=[]
train_loss = []
val_acc = []
val_recall=[]
val_precision=[]
val_F1=[]
val_loss = []

while batch_num < number_of_batches-1:
  #sessions are an arbitrary number of batches
  #model saved to drive after each session so progress isn't lost if runtime ends

  #read the last batch_num from drive file
  with open(train_index_path, "r") as file:
    content = file.read()
  if content != "":
    batch_num = int(content)
  else:
    batch_num = 999999999

  loaded_cnn = load_model(model_path,
                          custom_objects={
                          'weighted_bce_loss': weighted_bce_loss,
                          'f1_metric': f1_metric
                          })

  #train a handfule of batches in one go
  for i in range(batches_at_once):

    history = train_one_batch(loaded_cnn, batch_size, number_of_batches, batch_num, X_train, X_val, y_train, y_val)
    batch_num +=1

    train_acc.extend(history.history['accuracy'])
    train_loss.extend(history.history['loss'])
    train_recall.extend(history.history['recall'])
    train_precision.extend(history.history['precision'])
    train_F1.extend(history.history['f1_metric'])

    val_acc.extend(history.history['val_accuracy'])
    val_loss.extend(history.history['val_loss'])
    val_recall.extend(history.history['val_recall'])
    val_precision.extend(history.history['val_precision'])
    val_F1.extend(history.history['val_f1_metric'])

  #save model to drive, and record what batch number was left off on
  loaded_cnn.save(model_path)
  with open(train_index_path, "w") as file:
    file.write(str(batch_num))

  #save metrics history
  metrics_frame = pd.DataFrame({
    "Training Accuracy": train_acc,
    "Training Loss": train_loss,
    "Training Recall": train_recall,
    "Training Precision": train_precision,
    "Training F1": train_F1,
    "Validation Accuracy": val_acc,
    "Validation Loss": val_loss,
    "Validation Recall": val_recall,
    "Validation Precision": val_precision,
    "Validation F1": val_F1
  })
  metrics_frame.to_csv(train_metrics_history_path, mode="a", header=False, index=False)
  full_history = pd.read_csv(train_metrics_history_path)

  #illustrate metrics history each session
  clear_output(wait=True)
  fig, axes = plt.subplots(5, 1, figsize=(10, 20))

  axes[0].plot(full_history.index, full_history['Training Accuracy'], label="Training Accuracy", linewidth=2)
  axes[0].plot(full_history.index, full_history['Validation Accuracy'], label="Validation Accuracy", linewidth=2)
  axes[0].set_xlabel("Session")
  axes[0].set_ylabel("Value")
  axes[0].set_title("Accuracy Over Batches")
  axes[0].legend(title="Metrics")
  axes[0].grid(True)

  axes[1].plot(full_history.index, full_history['Training Loss'], label="Training Loss", linewidth=2)
  axes[1].plot(full_history.index, full_history['Validation Loss'], label="Validation Loss", linewidth=2)
  axes[1].set_xlabel("Session")
  axes[1].set_ylabel("Value")
  axes[1].set_title("Loss Over Batches")
  axes[1].legend(title="Metrics")
  axes[1].grid(True)

  axes[2].plot(full_history.index, full_history['Training Recall'], label="Training Recall", linewidth=2)
  axes[2].plot(full_history.index, full_history['Validation Recall'], label="Validation Recall", linewidth=2)
  axes[2].set_xlabel("Session")
  axes[2].set_ylabel("Value")
  axes[2].set_title("Recall Over Batches")
  axes[2].legend(title="Metrics")
  axes[2].grid(True)

  axes[3].plot(full_history.index, full_history['Training Precision'], label="Training Precision", linewidth=2)
  axes[3].plot(full_history.index, full_history['Validation Precision'], label="Validation Precision", linewidth=2)
  axes[3].set_xlabel("Session")
  axes[3].set_ylabel("Value")
  axes[3].set_title("Precision Over Batches")
  axes[3].legend(title="Metrics")
  axes[3].grid(True)

  axes[4].plot(full_history.index, full_history['Training F1'], label="Training F1", linewidth=2)
  axes[4].plot(full_history.index, full_history['Validation F1'], label="Validation F1", linewidth=2)
  axes[4].set_xlabel("Session")
  axes[4].set_ylabel("Value")
  axes[4].set_title("F1 Score Over Batches")
  axes[4].legend(title="Metrics")
  axes[4].grid(True)

  plt.tight_layout()
  plt.show()

avg_train_acc = np.mean(train_acc)
avg_train_loss = np.mean(train_loss)
avg_train_recall = np.mean(train_recall)
avg_train_precision = np.mean(train_precision)
avg_train_F1 = np.mean(train_F1)
avg_val_acc = np.mean(val_acc)
avg_val_loss = np.mean(val_loss)
avg_val_recall = np.mean(val_recall)
avg_val_precision = np.mean(val_precision)
avg_val_F1 = np.mean(val_F1)
print("average training accuracy:", avg_train_acc,
      "\naverage training loss:", avg_train_loss,
      "\naverage training recall:", avg_train_recall,
      "\naverage training precision:", avg_train_precision,
      "\naverage training F1:", avg_train_F1,
      "\naverage validation accuracy:", avg_val_acc,
      "\naverage validation loss", avg_val_loss,
      "\naverage validation recall:", avg_val_recall,
      "\naverage validation precision:", avg_val_precision,
      "\naverage validation F1:", avg_val_F1
)

loaded_cnn.save(model_path)
with open(train_index_path, "w") as file:
  file.write(str(batch_num))

In [ ]:
'''
train_index_path = "/content/drive/My Drive/plantVis3/model3trainingIndex.txt"
train_metrics_history_path = "/content/drive/My Drive/plantVis3/train_metrics_history2.csv"

columns = [
    "Training Accuracy",
    "Training Loss",
    "Training Recall",
    "Training Precision",
    "Training F1",
    "Validation Accuracy",
    "Validation Loss",
    "Validation Recall",
    "Validation Precision",
    "Validation F1"
]

# Create empty DataFrame with the same header
df = pd.DataFrame(columns=columns)

# Overwrite file
df.to_csv(train_metrics_history_path, index=False)

print("Cleared all rows; kept header only.")
with open(train_index_path, "w") as file:
  file.write(str(0))
#'''
